# Coarse-to-Fine 3D Brain Tumor Segmentation (Google Colab Master Runner)
**Course:** CS3006 Introduction to Data Science for Engineers (IIITDM Kancheepuram, Team 12)

This notebook runs the complete 3D Brain Tumor Segmentation pipeline on Google Colab GPU:
- Verifies GPU environment (NVIDIA Tesla T4)
- Clones latest cfseg codebase and runs pytest smoke test suite
- Downloads BraTS 2020 dataset from Kaggle via API
- Preprocesses up to 180 cases (brain z-score normalization)
- Trains Stage 1 CoarseNet (100 epochs, 0.36M params)
- Trains Stage 2 FineNet A0 Full Model (150 epochs, 9.27M params)
- Generates qualitative 4-panel visual overlays & training curve plots
- Evaluates Advanced Improvements: A (Generalization shifts), B (Adaptive margin), C (TTA)
- Bundles all results and trained model weights into downloadable zip files

**Estimated GPU Runtime:** ~2.2 hours total (well within Colab daily free session limits).

In [ ]:
# Step 1: Verify Hardware Accelerator
# Ensure GPU is enabled under Runtime -> Change runtime type -> T4 GPU
!nvidia-smi


In [ ]:
# Step 2: Clone cfseg Codebase & Install Dependencies
import os
!rm -rf /content/cfseg
!git clone https://github.com/pengeon1/idse-project.git /content/cfseg
%cd /content/cfseg

!pip install -q nibabel scipy matplotlib tabulate pandas pytest
!python -m pytest -p no:launch_testing tests/test_smoke.py -v


In [ ]:
# Step 3: Download BraTS 2020 Dataset from Kaggle
# You can obtain kaggle.json from Kaggle -> Profile icon (top-right) -> Settings -> API -> Create New Token
import os
from google.colab import files

kaggle_json = "/root/.kaggle/kaggle.json"
if not os.path.exists(kaggle_json):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p /root/.kaggle
    !mv kaggle.json /root/.kaggle/ 2>/dev/null || true
    !chmod 600 /root/.kaggle/kaggle.json

!mkdir -p /content/data
if not os.path.exists("/content/data/brats2020/BraTS2020_TrainingData"):
    print("Downloading BraTS 2020 dataset via Kaggle API (~3.6 GB)...")
    !kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content/data
    print("Extracting dataset...")
    !unzip -q /content/data/brats20-dataset-training-validation.zip -d /content/data/brats2020
    print("Dataset ready at /content/data/brats2020!")
else:
    print("Dataset already unpacked and ready!")


In [ ]:
# Step 4: Run Training & Baseline Evaluation (~2.2 hours on T4 GPU)
%cd /content/cfseg

!python scripts/run_kaggle_pipeline.py --work_dir /content --max_cases 180 --s1_epochs 100 --s2_epochs 150 --skip_ablations --batch_size 2


In [ ]:
# Step 5: Display Training Curves & Qualitative Visual Overlays
import os
from IPython.display import display, Image, Markdown

for img_name in ["curves_stage1.png", "curves_stage2_a0.png"]:
    p = os.path.join("/content/results", img_name)
    if os.path.exists(p):
        print(f"--- {img_name} ---")
        display(Image(p))

overlays_dir = "/content/results/overlays_a0"
if os.path.exists(overlays_dir):
    print("")
    print("--- Qualitative Visual Overlays (4 cases) ---")
    for f in sorted(os.listdir(overlays_dir)):
        if f.endswith(".png"):
            print(f"Displaying: {f}")
            display(Image(os.path.join(overlays_dir, f)))


In [ ]:
# Step 6: Run Advanced Improvements: A (Generalization), B (Adaptive Margin), C (TTA)
%cd /content/cfseg
!git fetch origin main -q 2>/dev/null && git reset --hard origin/main -q 2>/dev/null || true

!python scripts/run_advanced_eval.py --work_dir /content --split test

import os
from IPython.display import display, Markdown

if os.path.exists("/content/results/ADVANCED_RESULTS.md"):
    print("=" * 80)
    print("ADVANCED RESULTS (A, B, C) SUMMARY TABLE:")
    print("=" * 80)
    with open("/content/results/ADVANCED_RESULTS.md") as f:
        display(Markdown(f.read()))


In [ ]:
# Step 7: Download Results & Model Weights to Your Computer
from google.colab import files
import os

for bundle in ["/content/results_bundle.zip", "/content/advanced_results_bundle.zip"]:
    if os.path.exists(bundle):
        print(f"Triggering browser download for: {bundle}")
        files.download(bundle)
